In [1]:
import torch
from torch.utils.data import Dataset
from torch.utils.data.dataloader import DataLoader
from mingpt.utils import set_seed
set_seed(3407)

In [2]:
import random

def random_mul_instance_curriculum(n, k_active):
    """
    n=3 zawsze.
    k_active = 1 -> 00x
    k_active = 2 -> 0xx
    k_active = 3 -> xxx
    Wynik zawsze 2n cyfr (dla n=3 -> 6), padded zerami.
    """
    assert 1 <= k_active <= n

    def sample_number_digits():
        # np. n=3, k_active=2 => [0, d1, d2]
        prefix_zeros = [0] * (n - k_active)
        active = [random.randint(0, 9) for _ in range(k_active)]
        return prefix_zeros + active

    a = sample_number_digits()
    b = sample_number_digits()

    val_a = int(''.join(map(str, a)))
    val_b = int(''.join(map(str, b)))
    val_c = val_a * val_b

    str_c = str(val_c)
    str_c = (2*n - len(str_c)) * '0' + str_c  # zawsze 6 cyfr dla n=3
    c = [int(d) for d in str_c]

    return a + b + c  # długość 4n = 12


In [3]:
import torch
from torch.utils.data import Dataset

class MulCurriculumDataset(Dataset):
    def __init__(self, split, n=3, k_active=1):
        assert split in {'train', 'test'}
        self.split = split
        self.n = n
        self.k_active = k_active

    def __len__(self):
        return 10000

    def get_vocab_size(self):
        return 10

    def get_block_size(self):
        return 4 * self.n - 1  # 11 dla n=3

    def __getitem__(self, idx):
        while True:
            rmi = random_mul_instance_curriculum(self.n, self.k_active)
            h = hash(str(rmi[:2*self.n]))  # split zależny tylko od wejścia a+b
            inp_split = 'test' if h % 4 == 0 else 'train'
            if inp_split == self.split:
                break

        x = torch.tensor(rmi[:-1], dtype=torch.long)
        y = torch.tensor(rmi[1:], dtype=torch.long)

        y[:2*self.n - 1] = -1  # ignoruj loss na wejściu
        return x, y


In [4]:
train_dataset = MulCurriculumDataset("train", n=3, k_active=1)
test_dataset  = MulCurriculumDataset("test",  n=3, k_active=3)

In [5]:
# create a GPT instance
from mingpt.model import GPT

model_config = GPT.get_default_config()
model_config.model_type = "gpt-micro"
model_config.vocab_size = train_dataset.get_vocab_size()
model_config.block_size = train_dataset.get_block_size()

model = GPT(model_config)
print("Model:", model_config.model_type, "heads/layers/embd =", model_config.n_head, model_config.n_layer, model_config.n_embd)
print("block_size =", model_config.block_size, "vocab_size =", model_config.vocab_size)


number of parameters: 0.80M
Model: gpt-micro heads/layers/embd = 4 4 128
block_size = 11 vocab_size = 10


In [6]:
# create a Trainer object
from mingpt.trainer import Trainer

train_config = Trainer.get_default_config()
train_config.learning_rate = 3e-4
train_config.max_iters = 20000
train_config.num_workers = 0

trainer = Trainer(train_config, model, train_dataset)

running on device cpu


In [7]:
def batch_end_callback(trainer):
    it = trainer.iter_num

    # curriculum schedule (dostosuj progi pod swój budżet obliczeniowy)
    if it < 4000:
        train_dataset.k_active = 1
    elif it < 10000:
        train_dataset.k_active = 2
    else:
        train_dataset.k_active = 3

    if it % 100 == 0:
        print(
            f"iter_dt {trainer.iter_dt * 1000:.2f}ms; "
            f"iter {it}: train loss {trainer.loss.item():.5f}; "
            f"k_active={train_dataset.k_active}"
        )

trainer.set_callback("on_batch_end", batch_end_callback)

trainer.run()


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:692: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  warnings.warn(warn_msg)


iter_dt 0.00ms; iter 0: train loss 2.34224; k_active=1
iter_dt 54.28ms; iter 100: train loss 0.33449; k_active=1
iter_dt 55.08ms; iter 200: train loss 0.18047; k_active=1
iter_dt 51.02ms; iter 300: train loss 0.05464; k_active=1
iter_dt 53.65ms; iter 400: train loss 0.05291; k_active=1
iter_dt 58.05ms; iter 500: train loss 0.02241; k_active=1
iter_dt 52.66ms; iter 600: train loss 0.03790; k_active=1
iter_dt 64.70ms; iter 700: train loss 0.00780; k_active=1
iter_dt 52.65ms; iter 800: train loss 0.00932; k_active=1
iter_dt 51.86ms; iter 900: train loss 0.00520; k_active=1
iter_dt 50.67ms; iter 1000: train loss 0.00260; k_active=1
iter_dt 54.43ms; iter 1100: train loss 0.02200; k_active=1
iter_dt 60.25ms; iter 1200: train loss 0.00220; k_active=1
iter_dt 54.86ms; iter 1300: train loss 0.00132; k_active=1
iter_dt 57.75ms; iter 1400: train loss 0.01203; k_active=1
iter_dt 52.39ms; iter 1500: train loss 0.01249; k_active=1
iter_dt 84.43ms; iter 1600: train loss 0.00054; k_active=1
iter_dt 54

In [ ]:
# now let's perform some evaluation
model.eval()
None

In [8]:
def eval_mul_split(trainer, split, max_batches=50):
    dataset = {"train": train_dataset, "test": test_dataset}[split]
    n = dataset.n

    results = []
    loader = DataLoader(dataset, batch_size=100, num_workers=0, drop_last=False)

    model.eval()
    for b, (x, y) in enumerate(loader):
        if b >= max_batches:
            break

        x = x.to(trainer.device)
        y = y.to(trainer.device)

        inp = x[:, :2 * n]      # a(3)+b(3)
        sol = y[:, -2 * n :]    # c(6)

        cat = model.generate(inp, 2 * n, do_sample=False)
        sol_candidate = cat[:, -2 * n :]

        correct = (sol == sol_candidate).all(1).cpu()
        results.extend(correct.int().tolist())

    rt = torch.tensor(results, dtype=torch.float)
    print("%s final score: %d/%d = %.2f%% correct" % (split, rt.sum(), len(results), 100 * rt.mean()))
    return rt.sum()

with torch.no_grad():
    train_score = eval_mul_split(trainer, "train", max_batches=50)
    test_score  = eval_mul_split(trainer, "test",  max_batches=50)

train final score: 164/5000 = 3.28% correct
test final score: 180/5000 = 3.60% correct
